# Smart Mental Wellness Assistant — Multimodal Model & Pipeline Exploration

This Jupyter Notebook demonstrates the core machine learning, computer vision, NLP, safety, and multimodal fusion pipelines powering the **Smart Mental Wellness Assistant**.

### Pipeline Components:
1. **Dynamic Dataset Loading**: Hugging Face Hub dataset provenance (`SetFit/go_emotions`, `Jeneral/fer-2013`, `av9ash/CSSR-S_labelled_suicidewatch_posts_reddit`)
2. **Text Emotion Recognition**: DistilBERT model & taxonomy mapping (`Joy`, `Sadness`, `Anger`, `Anxiety`, `Neutral`)
3. **Facial Expression Analysis**: FacialEmotionCNN PyTorch model with OpenCV face detection
4. **Crisis Classifier**: C-SSRS severe distress & risk assessment subsystem (`LOW`, `MODERATE`, `HIGH`)
5. **Contextual Stressor Detection**: Category extraction (`Academic`, `Career`, `Relationships`, `Sleep`, etc.)
6. **Multimodal Fusion**: Weighted late fusion of text and vision emotion probability distributions
7. **Intervention & Feedback**: CBT & mindfulness recommendation engine with feedback adaptation

## 1. Environment Setup & Import Dependencies

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import cv2
from pathlib import Path

# Add backend to sys.path
repo_root = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "backend"))

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 2. Dataset Provenance & Dynamic Dataset Loading

In [ ]:
from app.ml.dataset_manager import dataset_manager

status = dataset_manager.get_status()
print("Dataset Manager Provenance Status:")
print(json.dumps(status, indent=2))

# Load text sample dynamic stats
text_dataset = dataset_manager.load_text_dataset()
print(f"\nGoEmotions Text Dataset loaded successfully!")
print(f"Splits available: {list(text_dataset.keys())}")
print(f"Train sample count: {len(text_dataset['train'])}")
print(f"Sample text entry: {text_dataset['train'][0]}")

## 3. Text Emotion Model (DistilBERT NLP Pipeline)

In [ ]:
from app.services.text_service import text_analyzer

sample_texts = [
    "I am so excited and happy about our project milestone today!",
    "I feel so overwhelmed and worried about tomorrow's exams.",
    "It drives me crazy when people don't follow through on promises.",
    "I feel lonely and sad lately, like nothing is going right."
]

for sample in sample_texts:
    result = text_analyzer.analyze(sample)
    print(f"Text: '{sample}'")
    print(f"  → Emotion: {result['emotion']} (Confidence: {result['confidence']:.2%})")
    print(f"  → Top Tokens: {result.get('attributions', [])[:3]}")
    print("-" * 50)

## 4. Computer Vision Facial Expression Model (FacialEmotionCNN)

In [ ]:
from app.services.vision_service import vision_analyzer, VisionAnalysisError

# Test face detection guard on blank image
blank_img = np.zeros((200, 200, 3), dtype=np.uint8)
_, encoded = cv2.imencode(".png", blank_img)

try:
    res = vision_analyzer.analyze_image(encoded.tobytes())
    print("CV Analysis Result:", res)
except VisionAnalysisError as vae:
    print(f"✓ Correctly caught face-detection error: {vae.reason} (Status: {vae.status_code})")

## 5. Crisis Subsystem & Risk Assessment

In [ ]:
from app.safety.crisis_classifier import crisis_classifier

test_cases = [
    "I had a great walk outside today and drank some tea.",
    "I'm stressed about exams but managing okay with breaks.",
    "I feel like giving up on everything and cannot see any way out."
]

for phrase in test_cases:
    assessment = crisis_classifier.assess(phrase)
    print(f"Phrase: '{phrase}'")
    print(f"  → Risk Level: {assessment['level']}")
    print(f"  → Flags: {assessment.get('matched_indicators', [])}")
    print(f"  → Override Triggered: {assessment.get('safety_override_triggered', False)}")
    print("-" * 50)

## 6. Stressor Detection Engine

In [ ]:
from app.safety.stressor import detect_stressors

input_str = "I haven't slept well in days because of my upcoming university finals."
stressor = detect_stressors(input_str)
print(f"Input Text: '{input_str}'")
print(f"Detected Stressor: {stressor['category']} (Intensity: {stressor['intensity']}, Confidence: {stressor['confidence']:.2f})")

## 7. Multimodal Emotion Fusion Engine

In [ ]:
from app.services.fusion_service import compute_fusion

text_distribution = {"Joy": 0.10, "Sadness": 0.10, "Anger": 0.05, "Anxiety": 0.70, "Neutral": 0.05}
vision_distribution = {"Joy": 0.05, "Sadness": 0.20, "Anger": 0.05, "Anxiety": 0.65, "Neutral": 0.05}

fused = compute_fusion(text_distribution, vision_distribution, text_weight=0.65, vision_weight=0.35)
print("Multimodal Fusion Result:")
print(f"  Primary Emotion: {fused['emotion']}")
print(f"  Fused Confidence: {fused['confidence']:.2%}")
print(f"  Modalities Present: {fused['modalities_used']}")
print(f"  Normalized Scores: {fused['normalized_scores']}")

## 8. Personalized CBT Interventions & Feedback Adaptation

In [ ]:
from app.services.intervention_service import intervention_service

rec = intervention_service.recommend_intervention(
    user_id="demo_user_01",
    emotion="Anxiety",
    stressor_category="Academic",
    severity="HIGH"
)
print("Recommended Intervention:")
print(f"  Title: {rec['title']}")
print(f"  Category: {rec['category']}")
print(f"  Description: {rec['description']}")
print(f"  Steps: {rec['steps']}")